In [3]:
import requests
import json
import os

def extraer_resenas_steam(app_id, num_comentarios=100):
    """Extrae reseñas de Steam y las guarda automáticamente en un archivo JSON."""
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    parametros = {
        'language': 'spanish',
        'filter': 'recent',
        'num_per_page': num_comentarios
    }

    try:
        # Realizar petición
        headers = {
    'User-Agent': 'FakeReviewsScraper/1.0 (Proyecto académico LP2 - Contacto: jimmy@example.com)'
    }
        respuesta = requests.get(url, params=parametros, headers=headers)
        respuesta.raise_for_status()
        datos = respuesta.json()

        # Definir ruta de guardado
        nombre_archivo = f"resenas_{app_id}.json"
        ruta_guardado = os.path.join(os.getcwd(), nombre_archivo)

        # Guardar archivo
        with open(ruta_guardado, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=4)

        print(f"✅ Proceso completado. Datos guardados en: {ruta_guardado}")

    except requests.exceptions.RequestException as e:
        print(f"❌ Error de red: {e}")
    except IOError as e:
        print(f"❌ Error al guardar el archivo: {e}")

if __name__ == "__main__":
    # Ejecutar para el ID especificado
    extraer_resenas_steam(4704690, 100)

✅ Proceso completado. Datos guardados en: c:\Users\jimmy\Downloads\Lp2final\resenas_4704690.json


In [4]:
import requests
import pandas as pd
import os

def procesar_resenas_steam(app_id, num_comentarios=100):
    """
    Extrae reseñas de Steam, las ordena por valoración y longitud,
    y las exporta a un archivo CSV.
    """
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    params = {
        'language': 'spanish',
        'filter': 'recent',
        'num_per_page': num_comentarios
    }

    try:
        # Petición a la API
        headers = {
    'User-Agent': 'FakeReviewsScraper/1.0 (Proyecto académico LP2 - Contacto: jimmy@example.com)'
    }
        respuesta = requests.get(url, params=params, headers=headers)
        respuesta.raise_for_status()
        data = respuesta.json()

        # Crear DataFrame y limpiar datos
        df = pd.DataFrame(data.get('reviews', []))

        # Extraer campos anidados del autor
        df['usuario_id'] = df['author'].apply(lambda x: x.get('steamid'))
        df['horas_jugadas'] = df['author'].apply(lambda x: x.get('playtime_forever', 0) / 60)

        # Crear indicadores de ordenamiento
        df['es_recomendado'] = df['voted_up'].astype(int)
        df['longitud_mensaje'] = df['review'].apply(len)

        # Ordenar: primero por recomendación (desc), luego por longitud del texto (desc)
        df_final = df.sort_values(
            by=['es_recomendado', 'longitud_mensaje'],
            ascending=[False, False]
        )

        # Seleccionar y renombrar columnas para el CSV final
        columnas = ['usuario_id', 'voted_up', 'horas_jugadas', 'longitud_mensaje', 'review']
        df_export = df_final[columnas].rename(columns={'voted_up': 'recomendado'})

        # Guardar en CSV
        ruta_csv = os.path.join(os.getcwd(), f"resenas_{app_id}_limpio.csv")
        df_export.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

        print(f"✅ Éxito: {len(df_export)} reseñas procesadas.")
        print(f"📂 Archivo guardado en: {ruta_csv}")

    except Exception as e:
        print(f"❌ Error al procesar: {e}")

if __name__ == "__main__":
    procesar_resenas_steam(4704690, 100)

✅ Éxito: 100 reseñas procesadas.
📂 Archivo guardado en: c:\Users\jimmy\Downloads\Lp2final\resenas_4704690_limpio.csv


In [6]:
import requests
import pandas as pd
import os
import re

def calcular_indice_ia(texto):
    """
    Asigna una puntuación del 1 al 10 basándose en heurísticas de lenguaje.
    """
    texto = str(texto).lower()
    score = 1

    # Lista de conectores y términos comunes en textos generados por IA
    indicadores = [
        r"\ben conclusión\b", r"\bademás\b", r"\bsin embargo\b",
        r"\bpor otro lado\b", r"\btransformar\b", r"\bexperiencia\b",
        r"\baspectos\b", r"\bfundamental\b", r"\bdetallado\b"
    ]

    # Sumar puntos por vocabulario estructurado
    for patron in indicadores:
        if re.search(patron, texto):
            score += 1

    # Sumar puntos por falta de errores (perfección gramatical sospechosa)
    # Los humanos tienden a usar abreviaciones o cometer errores tipográficos
    if len(texto) > 40 and not re.search(r"[^\w\s.,!¡¿?]", texto):
        score += 2

    return min(score, 10)

def generar_tabla_analisis(app_id, num_comentarios=100):
    url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
    params = {'language': 'spanish', 'filter': 'recent', 'num_per_page': num_comentarios}

    try:
        respuesta = requests.get(url, params=params)
        respuesta.raise_for_status()
        data = respuesta.json()

        # Crear DataFrame
        df = pd.DataFrame(data.get('reviews', []))

        # Limpieza de columnas clave
        df['usuario_id'] = df['author'].apply(lambda x: x.get('steamid'))
        df['horas_jugadas'] = df['author'].apply(lambda x: x.get('playtime_forever', 0) / 60)
        df['voto'] = df['voted_up'].map({True: 'Recomendado', False: 'No recomendado'})

        # --- Aplicar procedimiento de IA ---
        df['indice_ia'] = df['review'].apply(calcular_indice_ia)

        # Ordenar tabla para análisis (mayor índice de IA primero)
        df_final = df[['usuario_id', 'voto', 'horas_jugadas', 'review', 'indice_ia']]
        df_final = df_final.sort_values(by='indice_ia', ascending=False)

        # Guardar como nueva tabla
        ruta_archivo = os.path.join(os.getcwd(), "tabla_analisis_ia.csv")
        df_final.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')

        print(f"✅ Tabla generada con éxito.")
        print(f"📂 Ubicación: {ruta_archivo}")

    except Exception as e:
        print(f"❌ Error al crear la tabla: {e}")

if __name__ == "__main__":
    generar_tabla_analisis(4704690, 100)

✅ Tabla generada con éxito.
📂 Ubicación: c:\Users\jimmy\Downloads\Lp2final\tabla_analisis_ia.csv


In [7]:
def ajustar_indice_por_horas_jugadas():
    """
    Lee la tabla ya generada y ajusta el índice de IA basándose en las horas de juego.
    """
    ruta_archivo = os.path.join(os.getcwd(), "tabla_analisis_ia.csv")
    
    # Verificar que el archivo original se haya creado correctamente
    if not os.path.exists(ruta_archivo):
        return
        
    # Leer el CSV generado por tu código
    df = pd.read_csv(ruta_archivo)
    
    # Lógica de decisión por horas:
    def recalcular_ia(row):
        score = row['indice_ia']
        horas = row['horas_jugadas']
        
        # 1. Sospecha extrema: 0 horas jugadas (Reseña hecha sin tocar el juego)
        if horas == 0:
            score += 3
        # 2. Sospecha alta: Menos de 1 hora de juego
        elif horas < 1.0:
            score += 2
        # 3. Sospecha leve: Entre 1 y 3 horas de juego
        elif horas < 3.0:
            score += 1
        # 4. Factor humano: Si tiene más de 30 horas, es muy probable que sea humano
        elif horas > 30.0:
            score -= 2
            
        # Asegurarnos de que la puntuación se mantenga entre 1 y 10
        return min(max(score, 1), 10)
        
    # Aplicar la nueva lógica a la columna
    df['indice_ia'] = df.apply(recalcular_ia, axis=1)
    
    # Volver a ordenar la tabla para que los mayores índices de IA (ajustados) queden arriba
    df = df.sort_values(by='indice_ia', ascending=False)
    
    # Sobrescribir el archivo original
    df.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')
    print("Filtro adicional aplicado: Índice IA ajustado según las horas jugadas.")

# Ejecución automática al finalizar tu script
if __name__ == "__main__":
    ajustar_indice_por_horas_jugadas()

Filtro adicional aplicado: Índice IA ajustado según las horas jugadas.


In [9]:
import requests
import pandas as pd
import os
import time

def obtener_nombre_usuario(steam_id):
    """
    Obtiene el nombre de usuario de Steam a partir del Steam ID consultando el XML público.
    """
    # Nos aseguramos de que el ID sea una cadena de texto sin decimales
    steam_id_str = str(steam_id).split('.')[0] 
    
    try:
        url = f"https://steamcommunity.com/profiles/{steam_id_str}?xml=1"
        respuesta = requests.get(url, timeout=5)
        texto_xml = respuesta.text
        
        # Extraemos el nombre de la etiqueta <steamID>
        if "<steamID><![CDATA[" in texto_xml:
            inicio = texto_xml.find("<steamID><![CDATA[") + 18
            fin = texto_xml.find("]]></steamID>")
            return texto_xml[inicio:fin]
        elif "<steamID>" in texto_xml:
            inicio = texto_xml.find("<steamID>") + 9
            fin = texto_xml.find("</steamID>")
            return texto_xml[inicio:fin]
            
        return f"Perfil_Privado_{steam_id_str[-4:]}" # Por si no se puede acceder
    except Exception as e:
        return "Desconocido"

def agregar_nombres_usuarios_a_tabla():
    ruta_archivo = os.path.join(os.getcwd(), "tabla_analisis_ia.csv")
    
    if not os.path.exists(ruta_archivo):
        print(f"❌ No se encontró el archivo: {ruta_archivo}. Ejecuta las celdas anteriores primero.")
        return
        
    print("⏳ Leyendo archivo e identificando usuarios (esto tomará unos segundos)...")
    df = pd.read_csv(ruta_archivo)
    
    # Extraemos IDs únicos para no hacerle solicitudes repetidas a Steam
    ids_unicos = df['usuario_id'].dropna().unique()
    mapa_nombres = {}
    
    total = len(ids_unicos)
    for i, steam_id in enumerate(ids_unicos, 1):
        mapa_nombres[steam_id] = obtener_nombre_usuario(steam_id)
        
        # Feedback de progreso
        if i % 10 == 0 or i == total:
            print(f"  ➜ Procesados {i}/{total} perfiles...")
            
        time.sleep(0.3) # Pequeña pausa para no saturar los servidores de Steam y evitar bloqueos
        
    # Mapear los nombres a una nueva columna
    df['nombre_usuario'] = df['usuario_id'].map(mapa_nombres)
    
    # Reordenar las columnas para que el nombre quede junto al ID
    columnas = df.columns.tolist()
    columnas.remove('nombre_usuario')
    idx = columnas.index('usuario_id') + 1
    columnas.insert(idx, 'nombre_usuario')
    df = df[columnas]
    
    # Guardar los cambios en el mismo archivo CSV
    df.to_csv(ruta_archivo, index=False, encoding='utf-8-sig')
    print(f"✅ ¡Nombres añadidos con éxito!")
    print(f"📂 Archivo actualizado en: {ruta_archivo}")

    # Mostrar una vista previa de cómo quedó la tabla
    display(df[['usuario_id', 'nombre_usuario', 'voto', 'horas_jugadas', 'indice_ia']].head())

if __name__ == "__main__":
    agregar_nombres_usuarios_a_tabla()

⏳ Leyendo archivo e identificando usuarios (esto tomará unos segundos)...
  ➜ Procesados 10/100 perfiles...
  ➜ Procesados 20/100 perfiles...
  ➜ Procesados 30/100 perfiles...
  ➜ Procesados 40/100 perfiles...
  ➜ Procesados 50/100 perfiles...
  ➜ Procesados 60/100 perfiles...
  ➜ Procesados 70/100 perfiles...
  ➜ Procesados 80/100 perfiles...
  ➜ Procesados 90/100 perfiles...
  ➜ Procesados 100/100 perfiles...
✅ ¡Nombres añadidos con éxito!
📂 Archivo actualizado en: c:\Users\jimmy\Downloads\Lp2final\tabla_analisis_ia.csv


,usuario_id,nombre_usuario,voto,horas_jugadas,indice_ia
0,76561198791237314,imperio romano occidente,No recomendado,0.133333,5
1,76561197968703369,PiterPanda,No recomendado,0.983333,5
2,76561199235865408,antrusquireal901,No recomendado,0.683333,5
3,76561198442321373,reynolds,No recomendado,0.233333,5
4,76561198864745072,👑 QueenDiana 👑,No recomendado,0.583333,5
